In [22]:
import trafpy.generator as tpg
from trafpy.benchmarker import BenchmarkImporter

num_eps = 4

racks_dict, num_racks = {}, 2
eps_per_rack = int(num_eps/num_racks)
for rack in range(num_racks):
    racks_dict[rack] = [ep for ep in range(rack*eps_per_rack, (rack*eps_per_rack)+eps_per_rack)]
print(racks_dict)

ep_capacity = 100000000/8

net = tpg.gen_arbitrary_network(num_eps=num_eps, ep_capacity=ep_capacity, racks_dict=racks_dict)

print(net.graph.keys())
print(net.graph['topology_type'])

print("max_nw_capacity:", net.graph['max_nw_capacity'])
print("topology_type:", net.graph['topology_type'])
print("endpoints:", net.graph['endpoints'])
print("endpoint_label:", net.graph['endpoint_label'])
print("num_channels_per_link:", net.graph['num_channels_per_link'])
print("ep_link_capacity:", net.graph['ep_link_capacity'])
print("ep_link_port_capacity:", net.graph['ep_link_port_capacity'])
print("curr_nw_capacity_used:", net.graph['curr_nw_capacity_used'])
print("num_active_connections:", net.graph['num_active_connections'])
print("total_connections_blocked:", net.graph['total_connections_blocked'])
print("node_labels:", net.graph['node_labels'])
print("channel_names:", net.graph['channel_names'])
print("rack_to_ep_dict:", net.graph['rack_to_ep_dict'])
print("ep_to_rack_dict:", net.graph['ep_to_rack_dict'])


network_load_config = {'network_rate_capacity': net.graph['max_nw_capacity'], 
                       'ep_link_capacity': net.graph['ep_link_capacity'],
                       'target_load_fraction': 0.1}

importer = BenchmarkImporter(benchmark_version='v001', load_prev_dists=False)

dists = {}
#dists['node_dist'] = tpg.gen_uniform_node_dist(eps=net.graph['endpoints'])
#dists['interarrival_time_dist'] = {1.0: 1.0}
#dists['flow_size_dist'] = {1.0: 1.0}

dists = importer.get_benchmark_dists(benchmark_name='private_enterprise', 
                                                  eps=net.graph['endpoints'], 
                                                  racks_dict=net.graph['rack_to_ep_dict'])


print(list(dists.keys()))


jsd_threshold = 0.9


{0: [0, 1], 1: [2, 3]}
dict_keys(['endpoints', 'endpoint_label', 'num_channels_per_link', 'ep_link_capacity', 'ep_link_port_capacity', 'max_nw_capacity', 'curr_nw_capacity_used', 'num_active_connections', 'total_connections_blocked', 'node_labels', 'topology_type', 'channel_names', 'rack_to_ep_dict', 'ep_to_rack_dict'])
arbitrary_endpoints_4_chancap_12500000.0_channels_1
max_nw_capacity: 25000000.0
topology_type: arbitrary_endpoints_4_chancap_12500000.0_channels_1
endpoints: ['0', '1', '2', '3']
endpoint_label: None
num_channels_per_link: 1
ep_link_capacity: 12500000.0
ep_link_port_capacity: 6250000.0
curr_nw_capacity_used: 0
num_active_connections: 0
total_connections_blocked: 0
node_labels: [None]
channel_names: ['channel_1']
rack_to_ep_dict: {'0': ['0', '1'], '1': ['2', '3']}
ep_to_rack_dict: {'0': '0', '1': '0', '2': '1', '3': '1'}
load_prev_dist=False. Will re-generate dists with given network params and override any previously saved distributions.
Set to save benchmark private_en

In [23]:
from pathlib import Path
import gzip
import pickle
import time

path_to_data = 'data/'
Path(path_to_data).mkdir(exist_ok=True, parents=True)

print('Generating \'{}\' traffic demands for {} network'.format(dists, net.graph['topology_type']))
    
# get node, flow size, and flow inter-arrival time benchmark dists
#dists = dcn_dist
    
# generate traffic demands
demand_data = tpg.create_demand_data(eps=net.graph['endpoints'],
                                     node_dist=dists['node_dist'],
                                     flow_size_dist=dists['flow_size_dist'],
                                     interarrival_time_dist=dists['interarrival_time_dist'],
                                     network_load_config=network_load_config,
                                     jensen_shannon_distance_threshold=jsd_threshold)
                                   #  min_last_demand_arrival_time=2)
    
    # save demands as pickle file
    # filename = path_to_data+'{}_demand_data.pickle'.format(dcn)
    # with gzip.open(filename, 'wb') as f:
    #     pickle.dump(demand_data, f)

#tpg.save_data_as_csv(data=demand_data,path_to_save=path_to_data+'First_try_40.csv',overwrite=True)

Generating '{'node_dist': array([[0.        , 0.13370833, 0.07064583, 0.07064583],
       [0.13370833, 0.        , 0.05397917, 0.05472917],
       [0.07064583, 0.05397917, 0.        , 0.11629167],
       [0.07064583, 0.05472917, 0.11629167, 0.        ]]), 'flow_size_dist': {62100.0: 2e-05, 525.0: 0.007313333333333333, 25.0: 0.05076, 2450.0: 0.0015666666666666667, 12325.0: 0.00014, 75.0: 0.02982, 700.0: 0.006006666666666667, 1900.0: 0.0020466666666666667, 8325.0: 0.0003733333333333333, 475.0: 0.008306666666666667, 300.0: 0.012013333333333334, 2775.0: 0.00134, 6275.0: 0.0005466666666666667, 77275.0: 6.666666666666667e-06, 1100.0: 0.0035666666666666668, 225.0: 0.01512, 4475.0: 0.0007133333333333333, 1: 0.03418, 3650.0: 0.0009466666666666666, 275.0: 0.012013333333333334, 825.0: 0.004826666666666667, 200.0: 0.01566, 5825.0: 0.0005333333333333334, 1000.0: 0.00416, 2700.0: 0.0014933333333333333, 1125.0: 0.003766666666666667, 69125.0: 1.3333333333333333e-05, 6600.0: 0.00046, 89525.0: 6.6666666

Packed 6000 flows in 0.244 s | Node distribution Jensen Shannon distance from target achieved: 6.078252471430213e-09


In [24]:
tpg.save_data_as_csv(data=demand_data,path_to_save=path_to_data+'priv_10_100mbps_B.csv',overwrite=True)

Time to save data to data/priv_10_100mbps_B.csv: 0.023464202880859375 s
